In [1]:
# here is a mathematical expression that takes 3 inputs and produces one output
from math import sin, cos

def f(a, b, c):
  return -a**3 + sin(3*b) - 1.0/c + b**2.5 - a**0.5

print(f(2, 3, 4))

6.336362190988558


In [ ]:
# write the function df that returns the analytical gradient of f
# i.e. use your skills from calculus to take the derivative, then implement the formula
# if you do not calculus then feel free to ask wolframalpha, e.g.:
# https://www.wolframalpha.com/input?i=d%2Fda%28sin%283*a%29%29%29

def gradf(a, b, c):
  epsilon = 0.001
  _a = ["_a", None, { }, None, 0.0] # label, value, partial gradient lambda, children and gradient relative to 
  _b = ["_b", None, { }, None, 0.0]
  _c = ["_c", None, 0.0, None, 0.0]
  
  _a[1] = a
  _b[1] = b
  _c[1] = c
    

  # |D| is a new node
  _op1 = lambda x: -x**3.0
  d = ["D", _op1(_a[1]), 0.0, (_a, None)]
  # a -> d
  dd_da = lambda: (_op1(_a[1] + epsilon) - d[1]) / epsilon
  _a[2]["D"] = dd_da
  print(f"Value of {_a[0]}: {_a[1]:.5f} - Local gradient for [d in respect to _a]: {_a[2]}\n")

  # |E| (new node)
  _op2 = lambda x = _b[1]: x*3.0
  e = ["E", _op2(_b[1]), 0.0, (_b, None)]
  # b -> e
  de_db = lambda: (_op2(_b[1] + epsilon) - e[1]) / epsilon
  _b[2]["E"] = de_db
  print(f"Value of {_b[0]}: {_b[1]:.5f} - Local gradient for [e in respect to b]: {_b[2]}\n")

  # |E1| is a new node
  _op = lambda x = e[1]: sin(x)
  e1 = ["E1", _op(e[1]), 0.0, (e, None)]
  # e -> e1
  de1_de = (_op(e[1] + epsilon) - e1[1]) / epsilon
  print("de1_de = (_op(e[1] + epsilon) - e1[1]) / epsilon: ", de1_de)
  e[2] += de1_de
  print(f"Value of {e[0]}: {e[1]:.5f} - Local gradient for [e_1 in respect to e]: {e[2]:.8f}\n")

  # |F| is a new node
  f = ["F", d[1] + e1[1], 0.0, (d, e1)]
  # d -> f & e1 -> f
  d[2] += 1.0
  e1[2] += 1.0
  print(f"Value of {d[0]}: {d[1]:.5f} - Local gradient for [f in respect to d]: {d[2]:.8f}")
  print(f"Value of {e1[0]}: {e1[1]:.5f} - Local gradient for [f in respect to e1]: {e1[2]:.8f}\n")

  # |G1| is a new node
  _op = lambda x = _c[1]: x**-1.0
  g1 = ["G1", _op(_c[1]), 0.0, (_c, None)]
  # g1 -> _c
  dg1_dc = (_op(_c[1] + epsilon) - g1[1]) / epsilon
  _c[2] += dg1_dc
  print(f"Value of {_c[0]}: {_c[1]:.5f} - Local gradient for [f1 in respect to f]: {_c[2]:.8f}\n")

  # |G| is a new node
  _op = lambda x = g1[1]: 1.0*x
  g = ["G", _op(g1[1]), 0.0, (g1, None)]
  # g -> g1
  dg_dg1 = (_op(g1[1] + epsilon) - g[1]) / epsilon
  g1[2] += dg_dg1
  print(f"Value of {g1[0]}: {g1[1]:.5f} - Local gradient for [f1 in respect to f]: {g1[2]:.8f}\n")


  # |H| is a new node
  h = ["H", f[1] - g[1], 0.0, (f, g)] # is that right
  # h -> f & h -> g
  f[2] += 1.0
  g[2] += -1.0
  print(f"Value of {f[0]}: {f[1]:.5f} - Local gradient for [h in respect to f]: {f[2]:.8f}")
  print(f"Value of {g[0]}: {g[1]:.5f} - Local gradient for [h in respect to g]: {g[2]:.8f}\n")


  # (H + I)([K] - J)T_
  # |I| is a new node
  _op3 = lambda x = _b[1]: x**2.5
  i = ["I", _op3(_b[1]), 0.0, (_b, None)]
  # b -> g
  di_db = lambda: (_op3(_b[1] + epsilon) - i[1]) / epsilon
  _b[2]["I"] = di_db
  print(f"Value of {_b[0]}: {_b[1]:.5f} - Local gradient for [g in respect to b]: {_b[2]}\n")


  # |J| is a new node
  _op4 = lambda x: x**0.5
  j = ["J", _op4(_a[1]), 0.0, (_a, None)]
  # a -> i
  dj_da = lambda: (_op4(_a[1] + epsilon) - j[1]) / epsilon
  _a[2]["J"] = dj_da
  print(f"Value of {_a[0]}: {_a[1]:.5f} - Local gradient for [i in respect to _a]: {_a[2]}\n")


  # |K| is a new node
  K = ["K", h[1] + i[1], 0.0, (h, i)]
  # K -> h & K -> i
  h[2] += 1.0
  i[2] += 1.0
  print(f"Value of {h[0]}: {h[1]:.5f} - Local gradient for [K in respect to h]: {h[2]:.8f}")
  print(f"Value of {i[0]}: {i[1]:.5f} - Local gradient for [K in respect to i]: {i[2]:.8f}\n")

  # |T_| is a new node
  T_ = ["T_", K[1] - j[1], 1.0, (K, j)]
  # T_ -> K & T_ -> j
  K[2] += 1.0
  j[2] += -1.0
  print(f"Value of {K[0]}: {K[1]:.5f} - Local gradient for [T_ in respect to K]: {K[2]:.8f}")
  print(f"Value of {j[0]}: {j[1]:.5f} - Local gradient for [T_ in respect to i]: {j[2]:.8f}\n")


  print("\n-----")
  print(f"T_ = {T_}")
  print("-----")

  backprop(T_)

  # return [df/da, df/db, df/dc]
  return [_a[4], _b[4], _c[4]]


def backprop(root):
  children: tuple = root[3]
  if children is None or len(children) == 0:
    return

  print(root)
  left = children[0]
  right = children[1]

  grad = root[2]
  if left is not None:
    if left[0] in ['_a', '_b']:
      local_grad = left[2][root[0]]()
      chain_value = grad * local_grad
      left[4] = chain_value if left[4] == 0.0 else chain_value + left[4]

    else:
      left[2] = grad * left[2]
      backprop(left)

  if right is not None:
    if right[0] in ['_a', '_b']:
      local_grad = right[2][root[0]]()
      chain_value = grad * local_grad
      right[4] = chain_value if right[4] == 0.0 else chain_value + right[4]
    else:
      right[2] = grad * right[2]
      backprop(right)


# expected answer is the list of
ans = [-12.353553390593273, 10.25699027111255, 0.0625]
yours = gradf(2, 3, 4)
for dim in range(3):
  ok = 'OK' if abs(yours[dim] - ans[dim]) < 1e-5 else 'WRONG!'
  print(f"{ok} for dim {dim}: expected {ans[dim]}, yours returns {yours[dim]}")

Value of _a: 2.00000 - Local gradient for [d in respect to _a]: {'D': <function gradf.<locals>.<lambda> at 0x108467420>}

Value of _b: 3.00000 - Local gradient for [e in respect to b]: {'E': <function gradf.<locals>.<lambda> at 0x1084671a0>}

de1_de = (_op(e[1] + epsilon) - e1[1]) / epsilon:  -0.9113361692545952
Value of E: 9.00000 - Local gradient for [e_1 in respect to e]: -0.91133617

Value of D: -8.00000 - Local gradient for [f in respect to d]: 1.00000000
Value of E1: 0.41212 - Local gradient for [f in respect to e1]: 1.00000000

Value of _c: 4.00000 - Local gradient for [f1 in respect to f]: -0.06248438

Value of G1: 0.25000 - Local gradient for [f1 in respect to f]: 1.00000000

Value of F: -7.58788 - Local gradient for [h in respect to f]: 1.00000000
Value of G: 0.25000 - Local gradient for [h in respect to g]: -1.00000000

Value of _b: 3.00000 - Local gradient for [g in respect to b]: {'E': <function gradf.<locals>.<lambda> at 0x1084671a0>, 'I': <function gradf.<locals>.<lambda

TypeError: unsupported operand type(s) for -: 'dict' and 'float'